In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import shap
import pickle

# Load the dataset
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi',
    'temp', 'rh', 'wind', 'rain', 'area'
]
fires_dt = pd.read_csv('../../05_src/data/fires/forestfires.csv', header=0, names=columns)

# Separate features (X) and target (y)
X = fires_dt.drop('area', axis=1)
y = fires_dt['area']

# Identify numerical and categorical columns
numerical_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_features = ['month', 'day']

# Define the two preprocessors
# Preprocessor 1: Scaling numerical features and one-hot encoding categorical features
preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ]
)

# Preprocessor 2: Scaling numerical features with log transformation and one-hot encoding categorical features
preproc2 = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('log', FunctionTransformer(np.log1p, validate=True)),
            ('scale', StandardScaler())
        ]), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ]
)

# Define the baseline regressor (Ridge Regression)
baseline_regressor = Ridge()

# Define the advanced regressor (Random Forest Regressor)
advanced_regressor = RandomForestRegressor(random_state=42)

# Create the four pipelines
# Pipeline A: preproc1 + baseline_regressor
pipeline_a = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', baseline_regressor)
])

# Pipeline B: preproc2 + baseline_regressor
pipeline_b = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('regressor', baseline_regressor)
])

# Pipeline C: preproc1 + advanced_regressor
pipeline_c = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', advanced_regressor)
])

# Pipeline D: preproc2 + advanced_regressor
pipeline_d = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('regressor', advanced_regressor)
])

# Define hyperparameter grids for each pipeline
param_grid_a = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]
}

param_grid_b = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]
}

param_grid_c = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20]
}

param_grid_d = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20]
}

# Perform GridSearchCV for each pipeline
def perform_grid_search(pipeline, param_grid, X, y):
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X, y)
    return grid_search

grid_search_a = perform_grid_search(pipeline_a, param_grid_a, X, y)
grid_search_b = perform_grid_search(pipeline_b, param_grid_b, X, y)
grid_search_c = perform_grid_search(pipeline_c, param_grid_c, X, y)
grid_search_d = perform_grid_search(pipeline_d, param_grid_d, X, y)

# Evaluate each model using cross-validation and RMSE
def evaluate_model(model, X, y):
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-scores)
    return rmse_scores.mean()

rmse_a = evaluate_model(grid_search_a.best_estimator_, X, y)
rmse_b = evaluate_model(grid_search_b.best_estimator_, X, y)
rmse_c = evaluate_model(grid_search_c.best_estimator_, X, y)
rmse_d = evaluate_model(grid_search_d.best_estimator_, X, y)

# Select the best-performing model
best_model = min(
    (rmse_a, grid_search_a.best_estimator_),
    (rmse_b, grid_search_b.best_estimator_),
    (rmse_c, grid_search_c.best_estimator_),
    (rmse_d, grid_search_d.best_estimator_),
    key=lambda x: x[0]
)[1]

# Split the data into training and testing sets for final evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the best model on the training data
best_model.fit(X_train, y_train)

# Predict on the test data
y_pred = best_model.predict(X_test)

# Calculate RMSE on the test data
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Test RMSE: {test_rmse:.2f}')

# Explain the best model's predictions using SHAP
# Note: For tree-based models like RandomForestRegressor, use TreeExplainer
explainer = shap.Explainer(best_model['regressor'], best_model['preprocessing'].transform(X_train))
shap_values = explainer(best_model['preprocessing'].transform(X_test))

# Local explanation for a single prediction
shap.plots.waterfall(shap_values[0])

# Global explanation: feature importance
shap.plots.bar(shap_values)

# Save the best model to a pickle file
with open('best_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)


FileNotFoundError: [Errno 2] No such file or directory: '../../05_src/data/fires/forestfires.csv'